# PowerPlus — Data Processing & Feature Engineering

This notebook prepares the three PowerPlus data sources for electricity-demand forecasting:

1. **Minute-level electricity data** → daily electricity consumption in **kWh** (target).
2. **Household metadata** → household/appliance characteristics.
3. **City-wise weather data** → daily weather features.

The final output is a clean modeling dataset for the next milestone: **Decision Tree regression**.

Expected repository structure:

```text
Project-Electricity-Demand-Forecasting/
├── city-wise_house_dataset/
│   ├── Islamabad/
│   ├── Karachi/
│   ├── Lahore/
│   ├── Multan/
│   ├── Peshawar/
│   └── Skardu/
├── weather_dataset/
│   ├── Islamabad.csv
│   ├── Karachi.csv
│   ├── Lahore.csv
│   ├── Multan.csv
│   ├── Peshawar.csv
│   └── Skardu.csv
├── metadata_ultimate.xlsx
└── processed_data/
```

**OpenWeatherMap API is not required** for this notebook because historical weather CSV files are used.

In [1]:
# 1. IMPORT LIBRARIES

from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Libraries loaded.")

Libraries loaded.


In [2]:
# 2. FIND PROJECT ROOT AND DEFINE PATHS

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for p in candidates:
        if (p / "city-wise_house_dataset").exists():
            return p
    return Path.cwd()

PROJECT_ROOT = find_project_root()

ELECTRICITY_DIR = PROJECT_ROOT / "city-wise_house_dataset"
WEATHER_DIR = PROJECT_ROOT / "weather_dataset"
METADATA_FILE = PROJECT_ROOT / "metadata_ultimate.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "processed_data"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ELECTRICITY_DIR:", ELECTRICITY_DIR)
print("WEATHER_DIR:", WEATHER_DIR)
print("METADATA_FILE:", METADATA_FILE)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: .
ELECTRICITY_DIR: city-wise_house_dataset
WEATHER_DIR: weather_dataset
METADATA_FILE: metadata_ultimate.xlsx
PROCESSED_DIR: processed_data


In [3]:
# 3. CHECK FILE STRUCTURE

print("\nCity folders:")
if ELECTRICITY_DIR.exists():
    for p in sorted(ELECTRICITY_DIR.iterdir()):
        if p.is_dir():
            print(" -", p.name)
else:
    print("ERROR: city-wise_house_dataset not found.")

print("\nWeather files:")
if WEATHER_DIR.exists():
    for p in sorted(WEATHER_DIR.iterdir()):
        if p.is_file():
            print(" -", p.name)
else:
    print("ERROR: weather_dataset not found.")

print("\nMetadata file exists:", METADATA_FILE.exists())


City folders:
 - Islamabad
 - Karachi
 - Lahore
 - Multan
 - Peshawar
 - Skardu

Weather files:
 - Islamabad.csv
 - Karachi.csv
 - Lahore.csv
 - Multan.csv
 - Peshawar.csv
 - Skardu.csv

Metadata file exists: True


In [4]:
# 4. HELPER FUNCTIONS

def normalize_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

def find_column(df, candidates):
    normalized = {normalize_col(c): c for c in df.columns}
    for candidate in candidates:
        key = normalize_col(candidate)
        if key in normalized:
            return normalized[key]
    for candidate in candidates:
        key = normalize_col(candidate)
        for ncol, original in normalized.items():
            if key in ncol or ncol in key:
                return original
    return None

def read_csv_safely(path):
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception:
            pass
    raise ValueError(f"Could not read {path}")

def infer_house_from_filename(path):
    match = re.search(r"house\s*#?\s*(\d+)", path.stem, flags=re.I)
    if match:
        return f"House#{int(match.group(1))}"
    return path.stem

def infer_city_from_electricity_path(path):
    relative = path.relative_to(ELECTRICITY_DIR)
    return relative.parts[0] if len(relative.parts) >= 2 else None

print("Helper functions ready.")

Helper functions ready.


## Electricity target calculation

Your electricity files are minute-level and contain:

```text
datetime | Usage (kW)
```

If each row represents one minute and `Usage` is instantaneous power in kW:

**Daily energy (kWh) = sum of minute-level kW readings / 60**

The resulting target column will be:

```text
electricity_kwh
```

We also keep coverage information so incomplete days can be identified.

In [5]:
# 5. LOAD ALL MINUTE-LEVEL ELECTRICITY FILES

electricity_frames = []
electricity_errors = []

csv_files = sorted(ELECTRICITY_DIR.rglob("*.csv"))

print("Electricity CSV files found:", len(csv_files))

for file_path in csv_files:
    try:
        df = read_csv_safely(file_path)

        datetime_col = find_column(
            df, ["datetime", "date_time", "timestamp", "date"]
        )
        usage_col = find_column(
            df, ["usage_kw", "usage (kw)", "usage", "power_kw", "power"]
        )

        if datetime_col is None or usage_col is None:
            electricity_errors.append({
                "file": str(file_path),
                "reason": "datetime or usage column not found",
                "columns": str(list(df.columns))
            })
            continue

        out = pd.DataFrame({
            # format="mixed": midnight rows are written as bare dates ("2023-11-01") and
            # every other minute with a time. Without it pandas infers the format from
            # the first row, and errors="coerce" then silently drops every reading that
            # does not match -- in 63 of 69 files, all but the midnight reading of each day.
            "datetime": pd.to_datetime(df[datetime_col], format="mixed", errors="coerce"),
            "usage_kw": pd.to_numeric(df[usage_col], errors="coerce")
        })

        out["house"] = infer_house_from_filename(file_path)
        out["city"] = infer_city_from_electricity_path(file_path)
        out["source_file"] = file_path.name

        out = out.dropna(subset=["datetime", "usage_kw"])
        out = out.sort_values("datetime")

        electricity_frames.append(out)

    except Exception as e:
        electricity_errors.append({
            "file": str(file_path),
            "reason": str(e),
            "columns": ""
        })

if not electricity_frames:
    raise ValueError("No usable electricity files were loaded.")

electricity_minute = pd.concat(electricity_frames, ignore_index=True)

print("Minute-level rows:", len(electricity_minute))
display(electricity_minute.head())

Electricity CSV files found: 69


Minute-level rows: 35043713


,datetime,usage_kw,house,city,source_file
0,2023-11-01 00:00:00,0.63,House#41,Islamabad,islamabad_House41.csv
1,2023-11-01 00:01:00,0.64,House#41,Islamabad,islamabad_House41.csv
2,2023-11-01 00:02:00,0.63,House#41,Islamabad,islamabad_House41.csv
3,2023-11-01 00:03:00,0.63,House#41,Islamabad,islamabad_House41.csv
4,2023-11-01 00:04:00,0.52,House#41,Islamabad,islamabad_House41.csv


In [6]:
# 6. CLEAN MINUTE-LEVEL ELECTRICITY

negative_count = (electricity_minute["usage_kw"] < 0).sum()

electricity_minute.loc[
    electricity_minute["usage_kw"] < 0, "usage_kw"
] = np.nan

electricity_minute = electricity_minute.dropna(subset=["usage_kw"])

duplicate_count = electricity_minute.duplicated(
    subset=["city", "house", "datetime"]
).sum()

electricity_minute = electricity_minute.drop_duplicates(
    subset=["city", "house", "datetime"],
    keep="first"
)

print("Negative readings removed:", negative_count)
print("Duplicate timestamps removed:", duplicate_count)
print("Clean minute rows:", len(electricity_minute))

Negative readings removed: 0
Duplicate timestamps removed: 5346326
Clean minute rows: 29697387


In [7]:
# 7. CONVERT MINUTE ELECTRICITY TO DAILY KWH

electricity_minute["date"] = electricity_minute["datetime"].dt.normalize()

daily_electricity = (
    electricity_minute
    .groupby(["city", "house", "date"], as_index=False)
    .agg(
        electricity_kwh=("usage_kw", lambda x: x.sum() / 60.0),
        readings=("usage_kw", "size"),
        avg_power_kw=("usage_kw", "mean"),
        max_power_kw=("usage_kw", "max"),
        min_power_kw=("usage_kw", "min"),
    )
)

daily_electricity["expected_readings"] = 1440

daily_electricity["coverage_pct"] = (
    daily_electricity["readings"]
    / daily_electricity["expected_readings"]
    * 100
)

daily_electricity["low_coverage_flag"] = (
    daily_electricity["coverage_pct"] < 90
)

daily_electricity = daily_electricity.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

print("Daily electricity rows:", len(daily_electricity))
display(daily_electricity.head(20))

Daily electricity rows: 21057


,city,house,date,electricity_kwh,readings,avg_power_kw,max_power_kw,min_power_kw,expected_readings,coverage_pct,low_coverage_flag
0,Islamabad,House#41,2023-11-01,8.259667,1440,0.344153,1.41,0.0,1440,100.0,False
1,Islamabad,House#41,2023-11-02,11.801000,1440,0.491708,3.80,0.0,1440,100.0,False
2,Islamabad,House#41,2023-11-03,11.706667,1440,0.487778,2.84,0.0,1440,100.0,False
3,Islamabad,House#41,2023-11-04,10.956333,1440,0.456514,4.53,0.0,1440,100.0,False
4,Islamabad,House#41,2023-11-05,10.849667,1440,0.452069,3.08,0.0,1440,100.0,False
5,Islamabad,House#41,2023-11-06,10.996833,1440,0.458201,0.65,0.0,1440,100.0,False
6,Islamabad,House#41,2023-11-07,10.506333,1440,0.437764,0.68,0.0,1440,100.0,False
7,Islamabad,House#41,2023-11-08,10.798000,1440,0.449917,4.68,0.0,1440,100.0,False
8,Islamabad,House#41,2023-11-09,12.842167,1440,0.535090,4.49,0.0,1440,100.0,False
9,Islamabad,House#41,2023-11-10,16.350667,1440,0.681278,2.53,0.0,1440,100.0,False


In [8]:
# 8. SAVE CLEAN DAILY ELECTRICITY

daily_electricity_file = PROCESSED_DIR / "daily_electricity_clean.csv"
daily_electricity.to_csv(daily_electricity_file, index=False)

print("Saved:", daily_electricity_file)

Saved: processed_data/daily_electricity_clean.csv


## Household metadata

Metadata is household-level information. It will be joined using:

```text
city + house
```

Examples include residents, property area, floors, construction year and appliance counts.

In [9]:
# 9. LOAD METADATA

if not METADATA_FILE.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_FILE}")

metadata = pd.read_excel(METADATA_FILE)

metadata.columns = [
    re.sub(r"\s+", " ", str(c)).strip()
    for c in metadata.columns
]

print("Metadata shape:", metadata.shape)
print(metadata.columns.tolist())
display(metadata.head())

Metadata shape: (59, 46)
['House', 'City', 'Owner/Rented', 'No. of people (Temp+Perm)', 'No. of Permanent residents', 'No. of Children (0-13)', 'No. of Adults (14-60)', 'No. of Seniors (above 60)', 'No. of temporary residents', 'Property Area (Marla)', 'Covered Area', 'No of Floors', 'Floor of Residency', 'Build year of house', 'Wapda Connection type', 'Average Ceiling Height ft', 'Ceiling Type', 'Roof Type', 'Flooring Type', 'Interior Wall', 'Exterior Wall', 'No. of rooms', 'Room Dimensions', 'Kitchen', 'Number of Washrooms', 'Number of Stores', 'Doors Type', 'Air Conditioners', 'Air Coolers', 'Refrigerators', 'Washing Machines', 'LED Bulbs', 'Tube Lights', 'Celling Fans', 'Wall Fans', 'Stand Fans', 'Water Dispensers', 'Water Pumps', 'Electric Cooker', 'Electric heaters', 'Electric Irons', 'Sewing Machine', 'Microwave Ovens', 'Geysers', 'UPS', 'Other Electronic Devices']


,House,City,Owner/Rented,No. of people (Temp+Perm),No. of Permanent residents,No. of Children (0-13),No. of Adults (14-60),No. of Seniors (above 60),No. of temporary residents,Property Area (Marla),Covered Area,No of Floors,Floor of Residency,Build year of house,Wapda Connection type,Average Ceiling Height ft,Ceiling Type,Roof Type,Flooring Type,Interior Wall,Exterior Wall,No. of rooms,Room Dimensions,Kitchen,Number of Washrooms,Number of Stores,Doors Type,Air Conditioners,Air Coolers,Refrigerators,Washing Machines,LED Bulbs,Tube Lights,Celling Fans,Wall Fans,Stand Fans,Water Dispensers,Water Pumps,Electric Cooker,Electric heaters,Electric Irons,Sewing Machine,Microwave Ovens,Geysers,UPS,Other Electronic Devices
0,House#1,Lahore,Owner,9.0,9.0,4.0,7.0,2.0,0.0,5.0,5.000000,3,Whole,1987.0,Net Meter,10.0,Concrete,Cemented,Marble,Cemented,Painted,5.0,13*13,2.0,6.0,2.0,Wooden,4.0,0,2.0,2,50.0,0,10.0,0.0,0,1.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,0
1,House#2,Lahore,Owner,5.0,5.0,1.0,3.0,1.0,0.0,65.0,120.000000,5,Whole,2009.0,3 phase,11.0,Fall Celling,Cemented,Marble,Painted,Painted,10.0,15*15,2.0,4.0,NaN,Wooden,5.0,0,2.0,1,120.0,0,20.0,0.0,0,1.0,1.0,0.0,0.0,2.0,0.0,1.0,0.0,0.0,0
2,House#3,Lahore,Owner,6.0,4.0,3.0,3.0,0.0,2.0,5.0,5.000000,2,2F,2011.0,1 phase,11.0,Concrete,Cemented,Marble,Painted,Painted,5.0,11*11,1.0,2.0,1.0,Wooden,4.0,0,1.0,1,30.0,0,5.0,0.0,2,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0
3,House#4,Lahore,Owner,5.0,3.0,0.0,5.0,0.0,2.0,33.0,17.687424,2,Whole,2014.0,3 phase,9.5,Concrete,Cemented,Marble,Painted,Painted,9.0,15*12,2.0,5.0,2.0,Wooden,4.0,2,1.0,1,100.0,2,14.0,0.0,0,0.0,2.0,1.0,3.0,2.0,0.0,1.0,2.0,0.0,0
4,House#5,Lahore,Owner,5.0,5.0,0.0,5.0,0.0,0.0,10.0,10.000000,1,Whole,2003.0,1 phase,11.0,Concrete,Cemented,Marble,Painted,Painted,4.0,12*12,1.0,2.0,0.0,Wooden,2.0,2,1.0,1,20.0,7,8.0,0.0,0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0


In [10]:
# 10. STANDARDIZE METADATA KEYS

metadata_house_col = find_column(
    metadata, ["House", "House Name", "House#"]
)
metadata_city_col = find_column(
    metadata, ["City"]
)

print("House column:", metadata_house_col)
print("City column:", metadata_city_col)

if metadata_house_col is None or metadata_city_col is None:
    raise ValueError("House or City column could not be detected.")

metadata_clean = metadata.copy()

metadata_clean["house"] = metadata_clean[metadata_house_col].astype(str).str.strip()
metadata_clean["city"] = metadata_clean[metadata_city_col].astype(str).str.strip()

def standardize_house(value):
    value = str(value).strip()
    m = re.search(r"house\s*#?\s*(\d+)", value, flags=re.I)
    return f"House#{int(m.group(1))}" if m else value

metadata_clean["house"] = metadata_clean["house"].apply(standardize_house)

metadata_clean = metadata_clean.replace(
    ["#REF!", "#DIV/0!", "#VALUE!", "#N/A", "N/A", "NA", ""],
    np.nan
)

metadata_clean = metadata_clean.drop_duplicates(
    subset=["city", "house"], keep="first"
)

display(metadata_clean[["city", "house"]].head(20))

House column: House
City column: City


,city,house
0,Lahore,House#1
1,Lahore,House#2
2,Lahore,House#3
3,Lahore,House#4
4,Lahore,House#5
5,Lahore,House#6
6,Lahore,House#7
7,Lahore,House#8
8,Lahore,House#9
9,Lahore,House#10


## Weather aggregation

Weather is hourly, while the target is daily.

We convert weather to one row per:

```text
city + date
```

Examples:

- temperature → daily mean and maximum
- humidity → daily mean and maximum
- precipitation → daily sum
- wind speed → mean and maximum
- pressure → mean
- solar radiation → mean and maximum
- solar energy → sum
- UV index → maximum

In [11]:
# 11. WEATHER AGGREGATION FUNCTION

def aggregate_weather_file(file_path):
    df = read_csv_safely(file_path)
    df = df.dropna(axis=1, how="all")

    datetime_col = find_column(
        df, ["datetime", "date_time", "timestamp", "date"]
    )

    if datetime_col is None:
        raise ValueError(f"No datetime column in {file_path.name}")

    # format="mixed" for the same reason as the electricity loader: midnight rows are
    # bare dates, so inferring from the first row would keep only the midnight hour.
    df["datetime"] = pd.to_datetime(
        df[datetime_col], format="mixed", errors="coerce"
    )
    df = df.dropna(subset=["datetime"])
    df["date"] = df["datetime"].dt.normalize()

    candidates = {
        "temperature": ["temperature", "temp"],
        "humidity": ["humidity"],
        "dew": ["dew", "dew_point", "dewpoint"],
        "precipitation": ["precipitation", "precip", "rain"],
        "wind_speed": ["wind_speed", "windspeed", "wind speed"],
        "wind_direction": ["wind_direction", "winddirection", "wind direction"],
        "pressure": ["pressure"],
        "solar_radiation": ["solar_radiation", "solarradiation"],
        "solar_energy": ["solar_energy", "solarenergy"],
        "uv_index": ["uv_index", "uvindex", "uv"],
    }

    for standard_name, options in candidates.items():
        found = find_column(df, options)
        if found is not None:
            df[standard_name] = pd.to_numeric(
                df[found], errors="coerce"
            )

    agg = {}

    for col in [
        "temperature", "humidity", "dew",
        "wind_speed", "wind_direction",
        "pressure", "solar_radiation"
    ]:
        if col in df.columns:
            agg[f"{col}_mean"] = (col, "mean")

    for col in [
        "temperature", "humidity", "wind_speed",
        "solar_radiation", "uv_index"
    ]:
        if col in df.columns:
            agg[f"{col}_max"] = (col, "max")

    for col in ["precipitation", "solar_energy"]:
        if col in df.columns:
            agg[f"{col}_sum"] = (col, "sum")

    if not agg:
        raise ValueError(
            f"No recognized weather fields in {file_path.name}"
        )

    daily = df.groupby("date").agg(**agg).reset_index()
    daily["city"] = file_path.stem.strip()

    return daily

In [12]:
# 12. LOAD ALL WEATHER FILES

weather_frames = []
weather_errors = []

weather_files = sorted(WEATHER_DIR.glob("*.csv"))

print("Weather CSV files found:", len(weather_files))

for file_path in weather_files:
    try:
        weather_frames.append(
            aggregate_weather_file(file_path)
        )
    except Exception as e:
        weather_errors.append({
            "file": str(file_path),
            "reason": str(e)
        })

if not weather_frames:
    raise ValueError("No usable weather CSV files were loaded.")

daily_weather = pd.concat(weather_frames, ignore_index=True)

daily_weather = daily_weather.sort_values(
    ["city", "date"]
).reset_index(drop=True)

print("Daily weather rows:", len(daily_weather))
display(daily_weather.head(20))

Weather CSV files found: 6
Daily weather rows: 3114


,date,temperature_mean,humidity_mean,dew_mean,wind_speed_mean,wind_direction_mean,pressure_mean,solar_radiation_mean,temperature_max,humidity_max,wind_speed_max,solar_radiation_max,uv_index_max,precipitation_sum,solar_energy_sum,city
0,2023-07-01,30.120833,56.889583,20.141667,7.875000,177.745833,1000.441667,315.358333,36.3,76.50,16.9,919.0,9,0.000,27.2,Islamabad
1,2023-07-02,31.866667,54.744583,21.241667,8.008333,204.612500,1000.700000,303.495833,37.5,69.18,23.3,900.0,9,0.000,26.2,Islamabad
2,2023-07-03,32.341667,54.057083,21.362500,9.579167,134.345833,999.900000,292.708333,39.1,80.18,27.0,897.0,9,3.285,25.3,Islamabad
3,2023-07-04,29.187500,61.527917,20.825000,11.741667,108.650000,998.766667,246.762500,35.1,77.38,29.8,890.0,9,17.725,21.2,Islamabad
4,2023-07-05,26.537500,71.199583,20.804167,11.595833,116.637500,1000.287500,224.520833,29.2,85.35,29.7,824.0,8,3.192,19.3,Islamabad
5,2023-07-06,27.858333,67.455000,20.933333,5.445833,143.637500,1002.608333,252.329167,32.6,89.74,20.0,866.1,9,0.138,21.8,Islamabad
6,2023-07-07,25.704167,72.892083,20.258333,9.162500,157.658333,1004.345833,208.070833,29.4,88.59,26.1,857.2,9,26.802,18.2,Islamabad
7,2023-07-08,25.845833,71.675417,20.187500,7.433333,110.983333,1003.887500,217.562500,31.0,88.12,34.4,1067.0,10,1.100,18.8,Islamabad
8,2023-07-09,27.620833,68.900833,20.816667,8.191667,198.608333,1002.520833,308.937500,33.9,97.01,22.6,967.0,10,0.830,26.8,Islamabad
9,2023-07-10,29.108333,62.188750,20.412500,8.358333,215.320833,1002.866667,319.366667,34.6,90.82,20.0,981.0,10,1.766,27.7,Islamabad


In [13]:
# 13. SAVE DAILY WEATHER

daily_weather_file = PROCESSED_DIR / "daily_weather.csv"
daily_weather.to_csv(daily_weather_file, index=False)

print("Saved:", daily_weather_file)

Saved: processed_data/daily_weather.csv


# Merge the three datasets

### 1. Electricity + metadata

Join on:

```text
city + house
```

### 2. Add weather

Join on:

```text
city + date
```

This creates a household-day dataset containing:

**household characteristics + appliances + historical electricity + weather**

In [14]:
# 14. MERGE ELECTRICITY WITH METADATA

electricity_keys = daily_electricity[["city", "house"]].drop_duplicates()
metadata_keys = metadata_clean[["city", "house"]].drop_duplicates()

missing_metadata = (
    electricity_keys
    .merge(metadata_keys, on=["city", "house"], how="left", indicator=True)
)

missing_metadata = missing_metadata[
    missing_metadata["_merge"] == "left_only"
].drop(columns="_merge")

print("Household-city combinations without metadata:")
display(missing_metadata)

merged = daily_electricity.merge(
    metadata_clean,
    on=["city", "house"],
    how="left",
    suffixes=("", "_metadata")
)

print("After electricity + metadata:", merged.shape)

Household-city combinations without metadata:


,city,house


After electricity + metadata: (21057, 57)


In [15]:
# 15. ADD DAILY WEATHER

merged = merged.merge(
    daily_weather,
    on=["city", "date"],
    how="left"
)

merged = merged.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

print("After weather merge:", merged.shape)
display(merged.head())

After weather merge: (21057, 71)


,city,house,date,electricity_kwh,readings,avg_power_kw,max_power_kw,min_power_kw,expected_readings,coverage_pct,low_coverage_flag,House,City,Owner/Rented,No. of people (Temp+Perm),No. of Permanent residents,No. of Children (0-13),No. of Adults (14-60),No. of Seniors (above 60),No. of temporary residents,Property Area (Marla),Covered Area,No of Floors,Floor of Residency,Build year of house,Wapda Connection type,Average Ceiling Height ft,Ceiling Type,Roof Type,Flooring Type,Interior Wall,Exterior Wall,No. of rooms,Room Dimensions,Kitchen,Number of Washrooms,Number of Stores,Doors Type,Air Conditioners,Air Coolers,Refrigerators,Washing Machines,LED Bulbs,Tube Lights,Celling Fans,Wall Fans,Stand Fans,Water Dispensers,Water Pumps,Electric Cooker,Electric heaters,Electric Irons,Sewing Machine,Microwave Ovens,Geysers,UPS,Other Electronic Devices,temperature_mean,humidity_mean,dew_mean,wind_speed_mean,wind_direction_mean,pressure_mean,solar_radiation_mean,temperature_max,humidity_max,wind_speed_max,solar_radiation_max,uv_index_max,precipitation_sum,solar_energy_sum
0,Islamabad,House#41,2023-11-01,8.259667,1440,0.344153,1.41,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0,20.933333,69.213750,14.708333,7.616667,163.254167,1018.425000,151.004167,27.4,87.99,42.8,653.2,7,0.245,13.0
1,Islamabad,House#41,2023-11-02,11.801000,1440,0.491708,3.80,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0,19.895833,62.315833,11.987500,4.720833,114.350000,1016.562500,165.195833,26.6,81.98,7.6,654.2,7,0.329,14.2
2,Islamabad,House#41,2023-11-03,11.706667,1440,0.487778,2.84,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0,19.508333,61.763750,11.475000,5.650000,147.008333,1013.525000,173.904167,26.7,79.88,9.4,667.7,7,0.000,15.0
3,Islamabad,House#41,2023-11-04,10.956333,1440,0.456514,4.53,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0,19.350000,59.337083,10.541667,5.491667,126.116667,1014.308333,171.891667,26.6,79.15,13.1,670.4,7,0.000,14.8
4,Islamabad,House#41,2023-11-05,10.849667,1440,0.452069,3.08,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0,18.512500,57.937917,9.441667,4.620833,132.545833,1015.170833,173.462500,26.3,78.96,7.2,668.2,7,0.000,15.0


In [16]:
# 16. WEATHER MERGE QUALITY CHECK

weather_columns = [
    c for c in daily_weather.columns
    if c not in ["city", "date"]
]

if weather_columns:
    weather_missing = (
        merged[weather_columns]
        .isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .to_frame("missing_percent")
    )
    display(weather_missing)

print("Cities:", merged["city"].nunique())
print("Houses:", merged["house"].nunique())
print("Date range:", merged["date"].min(), "to", merged["date"].max())

,missing_percent
temperature_mean,0.0
humidity_mean,0.0
dew_mean,0.0
wind_speed_mean,0.0
wind_direction_mean,0.0
pressure_mean,0.0
solar_radiation_mean,0.0
temperature_max,0.0
humidity_max,0.0
wind_speed_max,0.0


Cities: 6
Houses: 59
Date range: 2023-07-16 00:00:00 to 2024-11-28 00:00:00


# Calendar feature engineering

Calendar features help the model learn repeating patterns:

- year
- month
- day
- day of week
- week of year
- day of year
- weekend
- season

These are predictors. The target remains:

```text
electricity_kwh
```

In [17]:
# 17. CREATE CALENDAR FEATURES

merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day
merged["day_of_week"] = merged["date"].dt.dayofweek
merged["week_of_year"] = merged["date"].dt.isocalendar().week.astype(int)
merged["day_of_year"] = merged["date"].dt.dayofyear
merged["is_weekend"] = (merged["day_of_week"] >= 5).astype(int)

def season_from_month(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Autumn"

merged["season"] = merged["month"].apply(season_from_month)

display(
    merged[
        ["date", "year", "month", "day_of_week",
         "is_weekend", "season"]
    ].head()
)

,date,year,month,day_of_week,is_weekend,season
0,2023-11-01,2023,11,2,0,Autumn
1,2023-11-02,2023,11,3,0,Autumn
2,2023-11-03,2023,11,4,0,Autumn
3,2023-11-04,2023,11,5,1,Autumn
4,2023-11-05,2023,11,6,1,Autumn


# Lag features

Lag features tell the model about the household's previous electricity demand.

Examples:

```text
lag_1_day_kwh  = yesterday
lag_7_day_kwh  = same household 7 days ago
lag_30_day_kwh = same household 30 days ago
```

Rolling averages summarize recent historical demand.

We use `shift(1)` before rolling calculations so the current day's target is not accidentally used as an input feature.

In [18]:
# 18. CREATE LAG FEATURES

merged = merged.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

group_cols = ["city", "house"]

for lag in [1, 2, 3, 7, 14, 30]:
    merged[f"lag_{lag}_day_kwh"] = (
        merged.groupby(group_cols)["electricity_kwh"]
        .shift(lag)
    )

shifted = (
    merged.groupby(group_cols)["electricity_kwh"]
    .shift(1)
)

shifted_group = shifted.groupby(
    [merged["city"], merged["house"]]
)

for window in [3, 7, 14, 30]:
    merged[f"rolling_{window}_day_avg_kwh"] = (
        shifted_group
        .transform(lambda x, w=window:
                   x.rolling(w, min_periods=1).mean())
    )

display(
    merged[
        ["city", "house", "date", "electricity_kwh",
         "lag_1_day_kwh", "lag_7_day_kwh",
         "lag_30_day_kwh", "rolling_7_day_avg_kwh"]
    ].head(40)
)

,city,house,date,electricity_kwh,lag_1_day_kwh,lag_7_day_kwh,lag_30_day_kwh,rolling_7_day_avg_kwh
0,Islamabad,House#41,2023-11-01,8.259667,NaN,NaN,NaN,NaN
1,Islamabad,House#41,2023-11-02,11.801000,8.259667,NaN,NaN,8.259667
2,Islamabad,House#41,2023-11-03,11.706667,11.801000,NaN,NaN,10.030333
3,Islamabad,House#41,2023-11-04,10.956333,11.706667,NaN,NaN,10.589111
4,Islamabad,House#41,2023-11-05,10.849667,10.956333,NaN,NaN,10.680917
5,Islamabad,House#41,2023-11-06,10.996833,10.849667,NaN,NaN,10.714667
6,Islamabad,House#41,2023-11-07,10.506333,10.996833,NaN,NaN,10.761694
7,Islamabad,House#41,2023-11-08,10.798000,10.506333,8.259667,NaN,10.725214
8,Islamabad,House#41,2023-11-09,12.842167,10.798000,11.801000,NaN,11.087833
9,Islamabad,House#41,2023-11-10,16.350667,12.842167,11.706667,NaN,11.236571


# Household and appliance features

The metadata already contains household and appliance variables.

We convert numeric-looking appliance fields to numeric values and create:

```text
house_age
total_appliance_count
```

Text fields such as Owner/Rented, ceiling type and WAPDA connection type are kept as categorical features. They will be one-hot encoded in the Decision Tree notebook.

In [19]:
# 19. CLEAN NUMERIC HOUSEHOLD/APPLIANCE FEATURES

numeric_candidates = [
    "No. of people (Temp+Perm)",
    "No. of Permanent residents",
    "No. of Children (0-13)",
    "No. of Adults (14-60)",
    "No. of Seniors (above 60)",
    "No. of temporary residents",
    "Property Area (Marla)",
    "Covered Area",
    "No of Floors",
    "Build year of house",
    "Average Ceiling Height ft",
    "No. of rooms",
    "Number of Washrooms",
    "Number of Stores",
    "Air Conditioners",
    "Air Coolers",
    "Refrigerators",
    "Washing Machines",
    "LED Bulbs",
    "Tube Lights",
    "Celling Fans",
    "Wall Fans",
    "Stand Fans",
    "Water Dispensers",
    "Water Pumps",
    "Electric Cooker",
    "Electric heaters",
    "Electric Irons",
    "Sewing Machine",
    "Microwave Ovens",
    "Geysers",
    "UPS",
    "Other Electronic Devices",
]

for col in numeric_candidates:
    if col in merged.columns:
        merged[col] = pd.to_numeric(
            merged[col], errors="coerce"
        )

if "Build year of house" in merged.columns:
    merged["house_age"] = (
        merged["year"] - merged["Build year of house"]
    ).clip(lower=0, upper=200)

appliance_candidates = [
    "Air Conditioners",
    "Air Coolers",
    "Refrigerators",
    "Washing Machines",
    "LED Bulbs",
    "Tube Lights",
    "Celling Fans",
    "Wall Fans",
    "Stand Fans",
    "Water Dispensers",
    "Water Pumps",
    "Electric Cooker",
    "Electric heaters",
    "Electric Irons",
    "Sewing Machine",
    "Microwave Ovens",
    "Geysers",
    "UPS",
    "Other Electronic Devices",
]

appliance_columns = [
    c for c in appliance_candidates if c in merged.columns
]

if appliance_columns:
    merged["total_appliance_count"] = (
        merged[appliance_columns]
        .fillna(0)
        .sum(axis=1)
    )

print("Appliance columns used:")
print(appliance_columns)

Appliance columns used:
['Air Conditioners', 'Air Coolers', 'Refrigerators', 'Washing Machines', 'LED Bulbs', 'Tube Lights', 'Celling Fans', 'Wall Fans', 'Stand Fans', 'Water Dispensers', 'Water Pumps', 'Electric Cooker', 'Electric heaters', 'Electric Irons', 'Sewing Machine', 'Microwave Ovens', 'Geysers', 'UPS', 'Other Electronic Devices']


In [20]:
# 20. BASIC CLEANING AND DUPLICATE CHECK

merged = merged.replace([np.inf, -np.inf], np.nan)

before = len(merged)

merged = merged.drop_duplicates(
    subset=["city", "house", "date"],
    keep="first"
)

print("Duplicate daily rows removed:", before - len(merged))

merged = merged.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

Duplicate daily rows removed: 0


In [21]:
# 21. DATA QUALITY REPORT

quality_report = pd.DataFrame({
    "column": merged.columns,
    "dtype": merged.dtypes.astype(str).values,
    "missing_count": merged.isna().sum().values
})

quality_report["missing_percent"] = (
    quality_report["missing_count"]
    / len(merged)
    * 100
)

quality_report = quality_report.sort_values(
    "missing_percent",
    ascending=False
)

display(quality_report.head(60))

,column,dtype,missing_count,missing_percent
84,lag_30_day_kwh,float64,1770,8.405756
36,Number of Stores,float64,1093,5.190673
83,lag_14_day_kwh,float64,826,3.922686
34,Kitchen,float64,730,3.466781
42,LED Bulbs,float64,730,3.466781
56,Other Electronic Devices,float64,703,3.338557
82,lag_7_day_kwh,float64,413,1.961343
45,Wall Fans,float64,365,1.733390
27,Ceiling Type,object,365,1.733390
55,UPS,float64,365,1.733390


# Model-ready dataset

Target:

```text
electricity_kwh
```

The dataset now contains:

### Historical electricity
- lag 1, 2, 3, 7, 14, 30 days
- rolling 3, 7, 14, 30 day averages

### Weather
- temperature
- humidity
- dew
- precipitation
- wind
- pressure
- solar radiation
- solar energy
- UV index

### Household
- residents
- property information
- construction year
- rooms
- appliances and electrical equipment

### Calendar
- month
- day of week
- weekend
- season

In [22]:
# 22. SAVE MASTER MERGED DATASET

master_file = PROCESSED_DIR / "powerplus_daily_merged.csv"

merged.to_csv(
    master_file,
    index=False
)

print("Saved master dataset:", master_file)
print("Shape:", merged.shape)

Saved master dataset: processed_data/powerplus_daily_merged.csv
Shape: (21057, 91)


In [23]:
# 23. CREATE MODEL DATASET

required_history = [
    "lag_1_day_kwh",
    "lag_7_day_kwh",
    "lag_30_day_kwh"
]

model_data = merged.dropna(
    subset=["electricity_kwh"] + required_history
).copy()

model_file = PROCESSED_DIR / "powerplus_model_data.csv"

model_data.to_csv(
    model_file,
    index=False
)

print("Saved model dataset:", model_file)
print("Shape:", model_data.shape)

Saved model dataset: processed_data/powerplus_model_data.csv
Shape: (19287, 91)


In [24]:
# 24. FINAL TARGET CHECK

TARGET = "electricity_kwh"

print("TARGET:", TARGET)
print("\nTarget statistics:")
display(model_data[TARGET].describe().to_frame())

print("\nModel data preview:")
display(model_data.head())

TARGET: electricity_kwh

Target statistics:


,electricity_kwh
count,19287.000000
mean,17.691071
std,16.874847
min,0.000000
25%,6.226309
50%,12.053833
75%,23.991772
max,167.481126



Model data preview:


,city,house,date,electricity_kwh,readings,avg_power_kw,max_power_kw,min_power_kw,expected_readings,coverage_pct,low_coverage_flag,House,City,Owner/Rented,No. of people (Temp+Perm),No. of Permanent residents,No. of Children (0-13),No. of Adults (14-60),No. of Seniors (above 60),No. of temporary residents,Property Area (Marla),Covered Area,No of Floors,Floor of Residency,Build year of house,Wapda Connection type,Average Ceiling Height ft,Ceiling Type,Roof Type,Flooring Type,Interior Wall,Exterior Wall,No. of rooms,Room Dimensions,Kitchen,Number of Washrooms,Number of Stores,Doors Type,Air Conditioners,Air Coolers,Refrigerators,Washing Machines,LED Bulbs,Tube Lights,Celling Fans,Wall Fans,Stand Fans,Water Dispensers,Water Pumps,Electric Cooker,Electric heaters,Electric Irons,Sewing Machine,Microwave Ovens,Geysers,UPS,Other Electronic Devices,temperature_mean,humidity_mean,dew_mean,wind_speed_mean,wind_direction_mean,pressure_mean,solar_radiation_mean,temperature_max,humidity_max,wind_speed_max,solar_radiation_max,uv_index_max,precipitation_sum,solar_energy_sum,year,month,day,day_of_week,week_of_year,day_of_year,is_weekend,season,lag_1_day_kwh,lag_2_day_kwh,lag_3_day_kwh,lag_7_day_kwh,lag_14_day_kwh,lag_30_day_kwh,rolling_3_day_avg_kwh,rolling_7_day_avg_kwh,rolling_14_day_avg_kwh,rolling_30_day_avg_kwh,house_age,total_appliance_count
30,Islamabad,House#41,2023-12-01,27.418667,1440,1.142444,4.25,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,14.650000,71.057083,9.145833,8.308333,154.850000,1020.112500,136.866667,21.1,87.10,19.8,547.0,5,0.046,11.7,2023,12,1,4,48,335,0,Winter,26.232167,24.780000,24.310833,24.809833,6.263333,8.259667,25.107667,25.355571,16.584429,14.290344,9.0,35.0
31,Islamabad,House#41,2023-12-02,23.328500,1440,0.972021,3.84,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,14.929167,67.769167,8.520833,6.379167,141.620833,1019.262500,139.120833,22.2,88.20,10.8,568.3,6,0.000,11.9,2023,12,2,5,48,336,1,Winter,27.418667,26.232167,24.780000,27.571667,12.616500,11.801000,26.143611,25.728262,18.095524,14.928978,9.0,35.0
32,Islamabad,House#41,2023-12-03,0.000000,1440,0.000000,0.00,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,14.375000,66.563750,7.820833,4.575000,113.304167,1018.891667,119.112500,20.9,84.63,9.0,524.0,5,0.000,10.5,2023,12,3,6,48,337,1,Winter,23.328500,27.418667,26.232167,26.366500,0.000000,11.706667,25.659778,25.122095,18.860667,15.313228,9.0,35.0
33,Islamabad,House#41,2023-12-04,22.002833,1440,0.916785,4.29,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,13.904167,66.430417,7.108333,5.075000,146.945833,1018.245833,139.566667,21.0,85.79,8.3,574.0,6,0.000,12.1,2023,12,4,0,49,338,0,Winter,0.000000,23.328500,27.418667,23.418000,4.939000,10.956333,16.915722,21.355452,18.860667,14.923006,9.0,35.0
34,Islamabad,House#41,2023-12-05,25.429667,1440,1.059569,4.40,0.0,1440,100.0,False,House#41,Islamabad,Owner,6.0,6.0,2.0,4.0,0.0,0.0,4.5,4.5,2,Whole,2014.0,1 phase,12.0,Cemented,Cemented,Marble,Painted,Painted,5.0,12*13,1.0,4.0,0.0,Wooden,2.0,0,1.0,1,20.0,0,5.0,2.0,0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,12.937500,67.145000,6.420833,4.125000,138.045833,1017.929167,140.525000,20.3,94.51,7.2,578.6,6,0.000,12.1,2023,12,5,1,49,339,0,Winter,22.002833,0

In [25]:
# 25. CITY SUMMARY

city_summary = (
    model_data
    .groupby("city")
    .agg(
        houses=("house", "nunique"),
        days=("date", "nunique"),
        records=("electricity_kwh", "size"),
        avg_daily_kwh=("electricity_kwh", "mean"),
        min_daily_kwh=("electricity_kwh", "min"),
        max_daily_kwh=("electricity_kwh", "max")
    )
    .reset_index()
)

display(city_summary)

,city,houses,days,records,avg_daily_kwh,min_daily_kwh,max_daily_kwh
0,Islamabad,10,396,3171,14.279683,0.000000,102.209117
1,Karachi,10,396,3333,23.499101,0.000000,136.971975
2,Lahore,10,355,3237,23.805326,0.000000,137.472152
3,Multan,10,384,3317,18.941755,0.000223,167.481126
4,Peshawar,10,366,3191,16.093004,0.000000,100.597518
5,Skardu,9,472,3038,8.678041,0.000000,81.780713


In [26]:
# 26. CREATE FEATURE LIST FOR NEXT NOTEBOOK

numeric_features = [
    c for c in [
        # Weather
        "temperature_mean", "temperature_max",
        "humidity_mean", "humidity_max",
        "dew_mean", "precipitation_sum",
        "wind_speed_mean", "wind_speed_max",
        "pressure_mean",
        "solar_radiation_mean", "solar_radiation_max",
        "solar_energy_sum", "uv_index_max",

        # Lag features
        "lag_1_day_kwh", "lag_2_day_kwh", "lag_3_day_kwh",
        "lag_7_day_kwh", "lag_14_day_kwh", "lag_30_day_kwh",
        "rolling_3_day_avg_kwh", "rolling_7_day_avg_kwh",
        "rolling_14_day_avg_kwh", "rolling_30_day_avg_kwh",

        # Calendar
        "year", "month", "day", "day_of_week",
        "week_of_year", "day_of_year", "is_weekend",

        # Household/appliances
        "No. of people (Temp+Perm)",
        "No. of Permanent residents",
        "No. of Children (0-13)",
        "No. of Adults (14-60)",
        "No. of Seniors (above 60)",
        "No. of temporary residents",
        "Property Area (Marla)",
        "Covered Area",
        "No of Floors",
        "Average Ceiling Height ft",
        "No. of rooms",
        "Number of Washrooms",
        "Number of Stores",
        "Air Conditioners", "Air Coolers",
        "Refrigerators", "Washing Machines",
        "LED Bulbs", "Tube Lights",
        "Celling Fans", "Wall Fans", "Stand Fans",
        "Water Dispensers", "Water Pumps",
        "Electric Cooker", "Electric heaters",
        "Electric Irons", "Sewing Machine",
        "Microwave Ovens", "Geysers", "UPS",
        "Other Electronic Devices",
        "house_age", "total_appliance_count"
    ]
    if c in model_data.columns
]

categorical_features = [
    c for c in [
        "city", "house", "Owner/Rented",
        "Wapda Connection type", "Ceiling Type",
        "Roof Type", "Flooring Type",
        "Interior Wall", "Exterior Wall",
        "Room Dimensions", "Kitchen", "Doors Type",
        "season", "Floor of Residency"
    ]
    if c in model_data.columns
]

feature_list = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "type": (
        ["numeric"] * len(numeric_features) +
        ["categorical"] * len(categorical_features)
    )
})

feature_file = PROCESSED_DIR / "powerplus_feature_list.csv"
feature_list.to_csv(feature_file, index=False)

display(feature_list)
print("Saved:", feature_file)

,feature,type
0,temperature_mean,numeric
1,temperature_max,numeric
2,humidity_mean,numeric
3,humidity_max,numeric
4,dew_mean,numeric
...,...,...
73,Room Dimensions,categorical
74,Kitchen,categorical
75,Doors Type,categorical
76,season,categorical


Saved: processed_data/powerplus_feature_list.csv


# 27. PROJECT MILESTONE STATUS

After this notebook runs successfully:

### Completed
- [x] Collect/import household metadata
- [x] Import minute-level electricity data
- [x] Clean electricity timestamps and values
- [x] Convert minute-level kW to daily kWh target
- [x] Import historical weather CSVs
- [x] Aggregate hourly weather to daily city-level features
- [x] Merge electricity + metadata
- [x] Merge weather + household-day records
- [x] Create calendar features
- [x] Create lag features
- [x] Create rolling historical demand features
- [x] Save master modeling dataset

### Next
- [ ] Train/test split by date
- [ ] Encode categorical variables
- [ ] Train Decision Tree Regressor
- [ ] Evaluate MAE, RMSE and R²
- [ ] Create 7-day forecast
- [ ] Create 30-day forecast
- [ ] Add future weather data/forecast if weather is used for future prediction
- [ ] Build consumer input/interface
- [ ] Present predicted electricity demand to the consumer